# 04 Build dataset
Cache of step 03 → one dataset in linked tables (participants, trainings, 5 s, questionnaire), with location, WBGT and heat dose per training; excluded participants removed → `dataset/`.

In [ ]:
import sys
sys.path.append("../src")

from heatwatch import dataset, heat_stress
from heatwatch.loading import participants as pp
from heatwatch.reports import overview, export

## 1. Combine
Garmin Connect + .fit of the same training merged, runs that follow each other within `merge_gap` (120 s) merged, race labelled (longest run on race day), 5-s windows, weekly / daily values.

In [ ]:
ds = dataset.build()
ds

In [ ]:
ds.trainings.head()

### Questionnaires
1 row per participant per race. Gender in the dataset comes from the questionnaire, otherwise from the device.

In [ ]:
ds.participants.gender_source.value_counts(dropna=False)

In [ ]:
ds.questionnaire.head()

## 2. Heat stress
WBGT (Liljegren, as KNMI) during each training and the heat dose (HD = duration × effort × WBGT above 14 °C), weather from Open-Meteo
(only the grid position and date are sent). Indoor runs: estimated gym value. Downloads are kept in `weather/`.
Set `[weather] enabled = false` in the settings to skip.

In [ ]:
ds = heat_stress.add(ds)
ds.trainings[["wbgt_mean", "wbgt_max", "heat_dose"]].describe()

In [ ]:
(ds.trainings.heat_dose > 0).value_counts().rename(index={True: "heat dose > 0", False: "0 or no dose"})

## 3. Data check
Per participant: read correctly (OK) or why not. Used for the exclusion reasons (step 4 and the report of notebook 05); not saved in the dataset.

In [ ]:
check = overview.data_check(ds.info, pp.find_folders(ds.info.study_id), ds.participants)
check.status.str.split(":").str[0].value_counts()

In [ ]:
check[check.status != "OK"]

## 4. Exclusions
Outcome per participant (rules: notebook 05, `[exclusion]` in settings.toml). Excluded participants are removed from every table; the reasons are in the report of notebook 05.

In [ ]:
ds, check = overview.apply_exclusions(ds, check)
ds

In [ ]:
overview.excluded(ds.outcomes)

## 5. Overview

In [ ]:
par, devices, last_14d = overview.overview_tables(ds.participants)
par

In [ ]:
devices

In [ ]:
last_14d

## 6. Save
`dataset/`: participants.csv, trainings.csv, questionnaire.csv, 5s/<study_id>.csv, HEATWATCH.xlsx, HEATWATCH.pkl

In [ ]:
export.save_dataset(ds)